In [ ]:
%pip install requests

In [18]:
import requests
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# Apparently we need to add the blueprint of the db so the notebook can understands it
Base = declarative_base()

class Country(Base):
    __tablename__ = 'Country'
    id = Column(Integer, primary_key=True)
    country = Column(String, nullable=False)
    doneym = Column(String, nullable=False)
    capital = Column(String, nullable=False)
    region = Column(String, nullable=False)
    language = Column(String, nullable=False)
    currency = Column(String, nullable=False)
    cities = relationship("Cities", back_populates="linked_country")

class Cities(Base):
    __tablename__ = 'Cities'
    id = Column(Integer, primary_key=True)
    city = Column(String, nullable=False)
    country_name = Column(String, nullable=False)
    country_id = Column(Integer, ForeignKey('Country.id'))
    linked_country = relationship("Country", back_populates="cities")
    activities = relationship("Activities", back_populates="linked_city")

class Activities(Base):
    __tablename__ = 'Activities'
    id = Column(Integer, primary_key=True)
    activities = Column(String, nullable=False)
    city_id = Column(Integer, ForeignKey('Cities.id'))
    linked_city = relationship("Cities", back_populates="activities")

# Connecting to the db and then opening a session
engine = create_engine('sqlite:///travel_planner.db', echo=False)
Session = sessionmaker(bind=engine)
session = Session()

# Connecting tot he API then fetching the data. I added all the column names from the tables inside the url
print("Fetching data from REST Countries API...")
url = "https://restcountries.com/v3.1/all?fields=name,demonyms,capital,region,languages,currencies"
response = requests.get(url)

# The safety net, in case the api doesn't work we will get an error
if response.status_code != 200:
    print(f"Api error: {response.status_code}.")
    api_data = [] 
else:
    api_data = response.json()
    if isinstance(api_data, dict):
        api_data = []

# Parse and Insert Data ---
if api_data:
    print(f"I found a total of:{len(api_data)} countries. Adding them to the db")
    added_count = 0

    for item in api_data:
        
        name = item.get('name', {}).get('common', 'Unknown')
        
        demonyms = item.get('demonyms', {}).get('eng', {})
        doneym = demonyms.get('m', 'Unknown')
        
        capitals = item.get('capital', [])
        capital = capitals[0] if capitals else 'None'
        
        region = item.get('region', 'Unknown')
        
        languages = item.get('languages', {})
        language = list(languages.values())[0] if languages else 'Unknown'
        
        currencies = item.get('currencies', {})
        currency = list(currencies.keys())[0] if currencies else 'Unknown'

        # To prevent duplicates
        existing_country = session.query(Country).filter_by(country=name).first()
        
        if not existing_country:
            new_country = Country(
                country=name,
                doneym=doneym,
                capital=capital,
                region=region,
                language=language,
                currency=currency
            )
            session.add(new_country)
            added_count += 1

    # Save and commit the data
    session.commit()
    print(f"I added:{added_count} to travel_planner.db.")

else:
    print("Api didn't work")

session.close()

Fetching data from REST Countries API...
I found a total of:250 countries. Adding them to the db
I added:250 to travel_planner.db.


In [19]:
import requests
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

# Connecting to the db and then open a session
engine = create_engine('sqlite:///travel_planner.db', echo=False)
Session = sessionmaker(bind=engine)
session = Session()

# --- 2. Fetch City Data ---
print("Fetching city data from CountriesNow API...")
url = "https://countriesnow.space/api/v0.1/countries"
response = requests.get(url)

if response.status_code != 200:
    print(f"error: {response.status_code}.")
    api_data = []
else:
    api_data = response.json().get('data', [])

# --- 3. Parse and Insert Data (The Fast Way) ---
if api_data:
    print("Got data, let's print it")
    
    # Trick recommended by the clanker: Fetch all existing cities into a Python set for instant O(1) lookups.
    # This prevents SQLite from running 140,000 individual searches and freezing up!
    existing_cities = {(c.city, c.country_id) for c in session.query(Cities).all()}
    
    new_cities_to_add = []
    countries_matched = 0

    for item in api_data:
        api_country_name = item.get('country')
        city_list = item.get('cities', [])
        
        # Match and parse
        our_country = session.query(Country).filter_by(country=api_country_name).first()
        
        if our_country:
            countries_matched += 1
            
            # Grabbing every city
            for city_name in city_list: 
                
                # CHecking for duplicates
                if (city_name, our_country.id) not in existing_cities:
                    
                    # Stage the new city
                    new_cities_to_add.append(
                        Cities(
                            city=city_name, 
                            country_name=our_country.country,
                            country_id=our_country.id
                        )
                    )
                    # Add it to our local set so we don't duplicate it in this loop
                    existing_cities.add((city_name, our_country.id))

    # Commit to the db
    if new_cities_to_add:
        print(f"Saving a total of:{len(new_cities_to_add)} cities")
        
        # add_all() is SQLAlchemy's ultra-fast bulk insert method
        session.add_all(new_cities_to_add)
        session.commit()
        print(f"It worked, we added:{len(new_cities_to_add)} cities across {countries_matched} countries.")
    else:
        print("No new cities needed to be added, database is already up to date!")

else:
    print("No data found to process.")

session.close()

Fetching city data from CountriesNow API...
Got data, let's print it
Saving a total of:75749 cities
It worked, we added:75749 cities across 219 countries.
